In [40]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION: Set the CSV file path for each (model, variant) pair
# ============================================================
CSV_CONFIG = {
    ("GPT-4o", "Minimal"): "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/investigate_undetected/investigation_GPT-4o_Minimal.csv",
    ("GPT-4o", "Method"): "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/investigate_undetected/investigation_GPT-4o_Method.csv",
    ("GPT-4o", "Class"): "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/investigate_undetected/investigation_GPT-4o_Class.csv",
    ("Qwen-480B", "Minimal"): "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/investigate_undetected/investigation_Qwen-480B_Minimal.csv",
    ("Qwen-480B", "Method"): "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/investigate_undetected/investigation_Qwen-480B_Method.csv",
    ("Qwen-480B", "Class"): "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/investigate_undetected/investigation_Qwen-480B_Class.csv",
    ("GPTOSS-120B", "Minimal"): "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/investigate_undetected/investigation_GPTOSS-120B_Minimal.csv",
    ("GPTOSS-120B", "Method"): "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/investigate_undetected/investigation_GPTOSS-120B_Method.csv",
    ("GPTOSS-120B", "Class"): "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/investigate_undetected/investigation_GPTOSS-120B_Class.csv",
}
# ============================================================


def analyze_file(filepath: str) -> dict:
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    df["hypothesis"] = df["hypothesis"].astype(str).str.strip().str.upper()
    df["custom_id"] = df["custom_id"].astype(str).str.strip()

    h1_ids = set(df.loc[df["hypothesis"] == "H1", "custom_id"])
    h2_ids = set(df.loc[df["hypothesis"] == "H2", "custom_id"])

    return {
        "total_rows": len(df),
        "h1_only": len(h1_ids - h2_ids),
        "h2_only": len(h2_ids - h1_ids),
        "both": len(h1_ids & h2_ids),
        "total_h1": len(h1_ids),
        "total_h2": len(h2_ids),
        "total_unique": len(h1_ids | h2_ids),
    }


def main():
    rows = []

    for (model, variant), filepath in CSV_CONFIG.items():
        if not Path(filepath).exists():
            print(f"WARNING: File not found for ({model}, {variant}): {filepath}")
            continue

        stats = analyze_file(filepath)
        rows.append({"model": model, "variant": variant, **stats})

    results = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("HYPOTHESIS DISTRIBUTION BY MODEL-VARIANT PAIR")
    print("=" * 90)
    print(results.to_string(index=False))

    print("\n" + "-" * 90)
    totals = results[["total_rows", "h1_only", "h2_only", "both", "total_h1", "total_h2", "total_unique"]].sum()
    print(f"{'TOTAL':<18} {totals.to_dict()}")
    print("-" * 90)

    # Save to CSV
    output_path = "hypothesis_analysis_results.csv"
    results.to_csv(output_path, index=False)
    print(f"\nResults saved to {output_path}")


if __name__ == "__main__":
    main()


HYPOTHESIS DISTRIBUTION BY MODEL-VARIANT PAIR
      model variant  total_rows  h1_only  h2_only  both  total_h1  total_h2  total_unique
     GPT-4o Minimal         205       11        9     1        12        10            21
     GPT-4o  Method         164       10        7     2        12         9            19
     GPT-4o   Class          58        3        1     0         3         1             4
  Qwen-480B Minimal         116       11        2     1        12         3            14
  Qwen-480B  Method         204       10        3     1        11         4            14
  Qwen-480B   Class         262        8        4     1         9         5            13
GPTOSS-120B Minimal          16        4        0     1         5         1             5
GPTOSS-120B  Method          30        7        1     1         8         2             9
GPTOSS-120B   Class          60        9        0     1        10         1            10

----------------------------------------------------

Creating a H2 investigation file

In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION: Set the CSV file path for each (model, variant) pair
# ============================================================
CSV_CONFIG = {
    ("GPT-4o", "Minimal"): "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/investigate_undetected/investigation_GPT-4o_Minimal.csv",
    ("GPT-4o", "Method"): "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/investigate_undetected/investigation_GPT-4o_Method.csv",
    ("GPT-4o", "Class"): "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/investigate_undetected/investigation_GPT-4o_Class.csv",
    ("Qwen-480B", "Minimal"): "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/investigate_undetected/investigation_Qwen-480B_Minimal.csv",
    ("Qwen-480B", "Method"): "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/investigate_undetected/investigation_Qwen-480B_Method.csv",
    ("Qwen-480B", "Class"): "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/investigate_undetected/investigation_Qwen-480B_Class.csv",
    ("GPTOSS-120B", "Minimal"): "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/investigate_undetected/investigation_GPTOSS-120B_Minimal.csv",
    ("GPTOSS-120B", "Method"): "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/investigate_undetected/investigation_GPTOSS-120B_Method.csv",
    ("GPTOSS-120B", "Class"): "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/investigate_undetected/investigation_GPTOSS-120B_Class.csv",
}
# ============================================================
OUTPUT_DIR = Path("/Volumes/Rachna-HD/RQResultsForPaper/RQ3/H2_manual")

def analyze_file(filepath: str) -> dict:
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    df["hypothesis"] = df["hypothesis"].astype(str).str.strip().str.upper()
    df["custom_id"] = df["custom_id"].astype(str).str.strip()

    h1_ids = set(df.loc[df["hypothesis"] == "H1", "custom_id"])
    h2_ids = set(df.loc[df["hypothesis"] == "H2", "custom_id"])

    return {
        "total_rows": len(df),
        "h1_only": len(h1_ids - h2_ids),
        "h2_only": len(h2_ids - h1_ids),
        "both": len(h1_ids & h2_ids),
        "total_h1": len(h1_ids),
        "total_h2": len(h2_ids),
        "total_unique": len(h1_ids | h2_ids),
    }





def export_h2_per_model():
    """Merge all variant CSVs per model, keep only H2 rows, save one CSV per model."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    models = {}
    for (model, variant), filepath in CSV_CONFIG.items():
        if not Path(filepath).exists():
            print(f"WARNING: File not found for ({model}, {variant}): {filepath}")
            continue
        df = pd.read_csv(filepath)
        df.columns = df.columns.str.strip()
        df["hypothesis"] = df["hypothesis"].astype(str).str.strip().str.upper()
        df["context_variant"] = variant  # ensure variant is tracked
        models.setdefault(model, []).append(df)

    for model, dfs in models.items():
        merged = pd.concat(dfs, ignore_index=True)
        h2_only = merged[merged["hypothesis"] == "H2"].drop_duplicates(subset=["custom_id", "context_variant"])
        out_path = OUTPUT_DIR / f"investigation_H2_{model}.csv"
        h2_only.to_csv(out_path, index=False)
        print(f"Saved {len(h2_only)} H2 rows for {model} → {out_path}")


def main():
    # ── Part 1: Hypothesis distribution analysis ──
    rows = []

    for (model, variant), filepath in CSV_CONFIG.items():
        if not Path(filepath).exists():
            print(f"WARNING: File not found for ({model}, {variant}): {filepath}")
            continue

        stats = analyze_file(filepath)
        rows.append({"model": model, "variant": variant, **stats})

    results = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("HYPOTHESIS DISTRIBUTION BY MODEL-VARIANT PAIR")
    print("=" * 90)
    print(results.to_string(index=False))

    print("\n" + "-" * 90)
    totals = results[["total_rows", "h1_only", "h2_only", "both", "total_h1", "total_h2", "total_unique"]].sum()
    print(f"{'TOTAL':<18} {totals.to_dict()}")
    print("-" * 90)

    output_path = "hypothesis_analysis_results.csv"
    results.to_csv(output_path, index=False)
    print(f"\nResults saved to {output_path}")

    # ── Part 2: Export merged H2 files per model ──
    print("\n" + "=" * 90)
    print("EXPORTING H2 INVESTIGATION FILES PER MODEL")
    print("=" * 90)
    export_h2_per_model()


if __name__ == "__main__":
    main()


HYPOTHESIS DISTRIBUTION BY MODEL-VARIANT PAIR
      model variant  total_rows  h1_only  h2_only  both  total_h1  total_h2  total_unique
     GPT-4o Minimal         205       11        9     1        12        10            21
     GPT-4o  Method         164       10        7     2        12         9            19
     GPT-4o   Class          58        3        1     0         3         1             4
  Qwen-480B Minimal         116       11        2     1        12         3            14
  Qwen-480B  Method         204       10        3     1        11         4            14
  Qwen-480B   Class         262        8        4     1         9         5            13
GPTOSS-120B Minimal          16        4        0     1         5         1             5
GPTOSS-120B  Method          30        7        1     1         8         2             9
GPTOSS-120B   Class          60       10        0     1        11         1            11

----------------------------------------------------